# Run faster-phowhisper on Colab or Kaggle

This notebook installs the Streamlit app with `uv`, starts an ngrok tunnel, and launches `app.py` so the UI can be opened from a public URL.

Before running:

- Add `NGROK_AUTHTOKEN` to Colab secrets or Kaggle secrets, or set it manually in the token cell.
- Optional: add `HF_TOKEN` if your Hugging Face access requires authentication.
- For the Faster-Whisper backend, provide a CTranslate2 PhoWhisper checkpoint such as `checkpoints/phowhisper-base-ct2`.
- For the Cascaded Encoder backend, provide a notebook export such as `checkpoints/cascaded_phowhisper_ckpt/step_500/cascaded_phowhisper.pt`.


## 1. Locate or clone the project

If this notebook is already inside the repository, the cell below uses the current project. In a fresh hosted runtime, set `GITHUB_REPO_URL` to your repository URL and rerun the cell.

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

GITHUB_REPO_URL = ""  # Example: "https://github.com/your-name/faster-phowhisper.git"

def running_on_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules

def running_on_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent, Path("/content/faster-phowhisper"), Path("/kaggle/working/faster-phowhisper")]
project_dir = next((path for path in candidate_roots if (path / "app.py").exists() and (path / "pyproject.toml").exists()), None)

if project_dir is None:
    if not GITHUB_REPO_URL:
        raise RuntimeError(
            "Project files were not found. Upload the repository to this runtime, "
            "or set GITHUB_REPO_URL and rerun this cell."
        )
    target = Path("/content/faster-phowhisper") if running_on_colab() else Path("/kaggle/working/faster-phowhisper")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(target)], check=True)
    project_dir = target

os.chdir(project_dir)
print("Project directory:", project_dir)
print("Python:", sys.version)
print("Platform:", platform.platform())
if sys.version_info < (3, 12):
    print("WARNING: pyproject.toml requests Python >=3.12,<3.13. Change the hosted runtime Python if dependency resolution fails.")


## 2. Install dependencies with uv

This respects `pyproject.toml`, including the PyTorch CUDA 12.8 index configured for `torch==2.11.0`.

In [ ]:
%pip install -q uv pyngrok

import subprocess

subprocess.run(["uv", "sync"], check=True)


## 3. Enable GPU defaults

Run this notebook on a GPU runtime. The cell sets Streamlit defaults so ASR model paths use CUDA when available: PhoWhisper Transformers, Faster-Whisper/CTranslate2, and Cascaded Encoder.


In [ ]:
import os
import subprocess

probe = r"""
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No CUDA GPU is available. In Colab/Kaggle, switch the accelerator to GPU and rerun from the install cell.")
"""
subprocess.run(["uv", "run", "python", "-c", probe], check=True)

os.environ["APP_DEFAULT_DEVICE"] = "cuda"
os.environ["APP_DEFAULT_COMPUTE_TYPE"] = "float16"
os.environ["APP_DEFAULT_ASR_BACKEND"] = "transformers"
print("APP_DEFAULT_DEVICE=", os.environ["APP_DEFAULT_DEVICE"])
print("APP_DEFAULT_COMPUTE_TYPE=", os.environ["APP_DEFAULT_COMPUTE_TYPE"])
print("APP_DEFAULT_ASR_BACKEND=", os.environ["APP_DEFAULT_ASR_BACKEND"])


## 4. Configure secrets

Recommended secret names:

- `NGROK_AUTHTOKEN`
- `HF_TOKEN` (optional)

The fallback manual strings are intentionally blank; fill them only inside a private notebook/runtime.

In [ ]:
import os

MANUAL_NGROK_AUTHTOKEN = ""
MANUAL_HF_TOKEN = ""

def read_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def read_kaggle_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

ngrok_token = os.getenv("NGROK_AUTHTOKEN") or read_colab_secret("NGROK_AUTHTOKEN") or read_kaggle_secret("NGROK_AUTHTOKEN") or MANUAL_NGROK_AUTHTOKEN
hf_token = os.getenv("HF_TOKEN") or read_colab_secret("HF_TOKEN") or read_kaggle_secret("HF_TOKEN") or MANUAL_HF_TOKEN

if ngrok_token:
    os.environ["NGROK_AUTHTOKEN"] = ngrok_token
else:
    raise RuntimeError("NGROK_AUTHTOKEN is required to expose Streamlit publicly via ngrok.")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

print("NGROK_AUTHTOKEN configured:", bool(os.getenv("NGROK_AUTHTOKEN")))
print("HF_TOKEN configured:", bool(os.getenv("HF_TOKEN")))


## 5. Check model checkpoint paths

The app can download Hugging Face models for the Transformers backend. Local checkpoint-backed modes need files mounted or copied into the runtime.

In [ ]:
from pathlib import Path

ct2_default = Path("checkpoints/phowhisper-base-ct2")
cascade_default = Path("checkpoints/cascaded_phowhisper_ckpt")

print("Faster-Whisper CT2 default:", ct2_default, "exists=", (ct2_default / "model.bin").exists())
print("Cascaded Encoder default:", cascade_default, "exists=", cascade_default.exists())

if not (ct2_default / "model.bin").exists():
    print("Faster-Whisper backend warning: upload or convert a CT2 PhoWhisper checkpoint before selecting that backend.")

cascade_exports = list(cascade_default.glob("step_*/cascaded_phowhisper.pt")) if cascade_default.exists() else []
if not cascade_exports:
    print("Cascaded Encoder warning: upload the notebook export before selecting that backend.")
else:
    print("Found cascaded exports:")
    for path in sorted(cascade_exports):
        print(" -", path)


## 6. Optional: convert PhoWhisper to CTranslate2

Set `RUN_CT2_CONVERSION = True` only when you want to create the Faster-Whisper checkpoint inside this runtime. This can take time and disk space.

In [ ]:
import subprocess
from pathlib import Path

RUN_CT2_CONVERSION = False
PHOWHISPER_MODEL_ID = "vinai/PhoWhisper-base"
CT2_OUTPUT_DIR = "checkpoints/phowhisper-base-ct2"
CT2_QUANTIZATION = "float16"  # Use "int8" for CPU-focused runtimes.

if RUN_CT2_CONVERSION:
    Path(CT2_OUTPUT_DIR).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "uv", "run", "ct2-transformers-converter",
            "--model", PHOWHISPER_MODEL_ID,
            "--output_dir", CT2_OUTPUT_DIR,
            "--copy_files", "tokenizer.json", "preprocessor_config.json",
            "--quantization", CT2_QUANTIZATION,
        ],
        check=True,
    )
else:
    print("Skipping CT2 conversion.")


## 7. Launch Streamlit with ngrok

This cell starts one ngrok tunnel in the notebook, stores it in `APP_PUBLIC_URL`, and launches Streamlit in the background. The app sidebar will reuse that URL.

In [ ]:
import os
import subprocess
import time
from pathlib import Path

from pyngrok import conf, ngrok

PORT = 8501
LOG_FILE = Path("streamlit_colab_kaggle.log")

try:
    ngrok.kill()
except Exception:
    pass

conf.get_default().auth_token = os.environ["NGROK_AUTHTOKEN"]
public_url = ngrok.connect(PORT, "http").public_url
os.environ["APP_PUBLIC_URL"] = public_url
os.environ["STREAMLIT_SERVER_PORT"] = str(PORT)
os.environ.setdefault("APP_DEFAULT_DEVICE", "cuda")
os.environ.setdefault("APP_DEFAULT_COMPUTE_TYPE", "float16")
os.environ.setdefault("APP_DEFAULT_ASR_BACKEND", "transformers")

print("Public URL:", public_url)

log_handle = LOG_FILE.open("w", encoding="utf-8")
streamlit_process = subprocess.Popen(
    [
        "uv", "run", "streamlit", "run", "app.py",
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

time.sleep(5)
print("Streamlit process id:", streamlit_process.pid)
print("Open the public URL above. Logs are written to", LOG_FILE)


## 8. View logs


In [ ]:
from pathlib import Path

LOG_FILE = Path("streamlit_colab_kaggle.log")
if LOG_FILE.exists():
    lines = LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-80:]))
else:
    print("No log file yet.")


## 9. Stop the app


In [ ]:
try:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
    print("Stopped Streamlit.")
except NameError:
    print("No streamlit_process variable found in this notebook session.")
except Exception as exc:
    print("Could not stop Streamlit cleanly:", exc)

try:
    ngrok.kill()
    print("Stopped ngrok.")
except Exception as exc:
    print("Could not stop ngrok cleanly:", exc)
